# Unix-terminal. Работа с файлами

## Мотивация

Почти вся работа на вычислительных серверах и кластерах идёт через терминал: там нет графического интерфейса, а запуск обучения, просмотр логов, копирование датасетов и настройка окружения делаются командами. Уверенное владение файлами, потоками, правами и процессами экономит часы и защищает от потери данных — например, от случайного `rm` не того каталога.

Сегодня соберём базовые кирпичики Unix-терминала, на которых держится вся дальнейшая работа с сервером.

## 1. `echo`, стандартные потоки, перенаправления и системные переменные

### `echo`

`echo` выводит аргументы через пробел и добавляет перевод строки. Опция `-n` убирает перевод строки. Поведение `-e` и обратных слешей может различаться между оболочками, поэтому здесь `echo` используется для простого текста.

Переменные заключают в двойные кавычки: `echo "$student_full_name"`. Кавычки сохраняют пробелы и отключают разбиение значения на аргументы. Фигурные скобки отделяют составное имя от соседнего текста: `echo "${student_group_name}_report.txt"`.

Имя переменной состоит из латинских букв, цифр и `_`, но не начинается с цифры. Дефисы, точки и пробелы в имени запрещены. Часто shell-переменные пишут в `snake_case`, а переменные окружения — в `UPPER_SNAKE_CASE`.

In [ ]:
%%bash
student_full_name='Linux Student'
student_group_name='ML 01'
echo "Hello, $student_full_name"
echo "report=${student_group_name}_report.txt"
echo -n 'without newline: '
echo 'continued on the same line'

### stdin, stdout и stderr

У процесса есть три стандартных потока:

- stdin (`0`) — ввод;
- stdout (`1`) — обычный результат;
- stderr (`2`) — ошибки и диагностика.

`>` перезаписывает файл, `>>` дописывает, `<` берёт stdin из файла, `2>` сохраняет ошибки отдельно, `2>&1` направляет stderr туда же, куда уже направлен stdout. Перенаправления обрабатываются слева направо.

### Heredoc: `<<EOF`

Heredoc передаёт команде сразу несколько строк ввода. После `<<` указывается маркер окончания; имя `EOF` — только распространённое соглашение, вместо него можно выбрать другое слово. Закрывающий маркер должен стоять один на строке без пробелов.

В `cat > file <<EOF` переменные и подстановки внутри блока раскрываются текущей оболочкой. В `cat > file <<'EOF'` содержимое записывается буквально: `$USER`, обратные кавычки и `$(command)` не исполняются. Для создания Bash-скрипта обычно нужен quoted marker `<<'EOF'`, чтобы переменные раскрылись позже, во время запуска созданного скрипта.

#### ❓ **Вопрос**: Чем `>` отличается от `2>` и что делает `2>&1`?

<details>

<summary><strong>Ответ</strong></summary>

`>` перенаправляет stdout (поток `1`), `2>` — stderr (поток `2`). Запись `2>&1` направляет stderr туда же, куда в этот момент указывает stdout. Порядок важен: `> file 2>&1` объединяет оба потока в файл, а `2>&1 > file` оставит stderr в терминале.

</details>

In [ ]:
%%bash
work_dir="${TMPDIR:-/tmp}/seminar1-streams"
mkdir -p "$work_dir"
echo 'first line' > "$work_dir/input.txt"
echo 'second line' >> "$work_dir/input.txt"
cat < "$work_dir/input.txt"
ls /etc/os-release /missing-file > "$work_dir/stdout.txt" 2> "$work_dir/stderr.txt" || true
ls /etc/os-release /missing-file > "$work_dir/all.txt" 2>&1 || true
cat > "$work_dir/heredoc.txt" <<'EOF'
This text has several lines.
$USER remains literal because EOF is quoted.
EOF
echo 'stdout:'; cat "$work_dir/stdout.txt"
echo 'stderr:'; cat "$work_dir/stderr.txt"

### Системные переменные

Полезные переменные окружения: `$USER`, `$HOME`, `$SHELL`, `$PATH`, `$PWD`, `$OLDPWD`, `$LANG`. Полный вывод `env` нельзя публиковать: среди переменных могут быть токены.

Обычная shell-переменная видна текущему Bash. После `export` она передаётся дочерним процессам. `bash -c 'КОМАНДЫ'` запускает дочерний Bash и выполняет строку после `-c`. В `bash -c '...'` одинарные кавычки оставляют `$VARIABLE` для раскрытия дочерней оболочкой. Аргументы передаются как `bash -c 'echo "$1"' bash value`: слово `bash` становится `$0`, следующий аргумент — `$1`.

#### ❓ **Вопрос**: Почему дочерний `bash -c` не видит переменную до `export`?

<details>

<summary><strong>Ответ</strong></summary>

Обычная переменная принадлежит только текущей оболочке. В дочерний процесс передаются лишь переменные окружения, помеченные `export`. Поэтому до `export COURSE_NAME` дочерний `bash -c` печатает `missing`, а после — значение переменной.

</details>

In [ ]:
%%bash
echo "USER=${USER}"
echo "REPORT=${USER}_report.txt"
echo "HOME=$HOME"
echo "SHELL=$SHELL"
echo "PWD=$PWD"
COURSE_EXECUTION_MODE='development'
bash -c 'echo "before export: ${COURSE_EXECUTION_MODE:-missing}"'
export COURSE_EXECUTION_MODE
bash -c 'echo "after export: $COURSE_EXECUTION_MODE"'
bash -c 'echo "first=$1 second=$2"' bash 'one value' 'two value'

## 2. Работа с папками и файлами. `touch` и условия

`pwd` показывает текущий каталог. `cd` меняет его, `cd ..` поднимается выше, `cd -` возвращает предыдущий. `mkdir -p` создаёт дерево каталогов. `touch` создаёт пустой файл или обновляет время существующего. `cp` копирует файл, `cp -r` — каталог с содержимым, `mv` перемещает или переименовывает, `cat` выводит небольшой файл.

### Удаление

`rm file` удаляет файл без помещения в корзину. `rmdir directory` удаляет только пустой каталог. `rm -r directory` рекурсивно удаляет каталог со всем содержимым. Опция `-f` отключает запросы и игнорирует отсутствующие файлы. Поэтому `rm -rf` особенно опасен: рекурсивно и без подтверждения удаляет всё по указанному пути, а ошибка в переменной, текущем каталоге или пробеле может выбрать не те данные.

Перед рекурсивным удалением проверяют `pwd`, выводят точный путь через `echo`, просматривают содержимое и убеждаются, что переменная не пустая. Пути заключают в кавычки, а `--` отделяет опции от имени: `rm -- "$file_name"`. Не выполняйте `rm -rf` с `/`, `$HOME`, пустой переменной, wildcard `*` или непроверенным пользовательским вводом.

### Условия

Команда `first || second` запускает `second`, только если `first` завершилась с ошибкой. Команда `first && second` запускает `second` только после успеха. Код завершения последней команды хранится в `$?`. Проверки `test -e`, `test -f`, `test -d` определяют существование пути, обычного файла и каталога.

In [ ]:
%%bash
set -e
demo_dir="${TMPDIR:-/tmp}/seminar1-files"
mkdir -p "$demo_dir"/{input,work,result}
cd "$demo_dir"
touch input/empty.txt
echo 'data' > 'input/file with spaces.txt'
cp input/empty.txt work/
mv work/empty.txt work/renamed.txt
test -f work/renamed.txt && echo 'file exists'
test -f work/missing.txt || echo 'file is missing'
touch result/remove-me.txt
echo "removing: $demo_dir/result/remove-me.txt"
rm -- result/remove-me.txt
mkdir -p result/empty-directory
rmdir result/empty-directory
test ! -e result/remove-me.txt && echo 'file removed'
cd input
pwd
cd -

## 3. Пользователи, группы и права

`whoami` показывает текущего пользователя, `id` — UID, основной GID и группы, `groups` — список групп. Запись `-rw-r-----` читается как тип объекта и три тройки прав: владелец, группа, остальные. `r` — чтение, `w` — запись, `x` — выполнение или вход в каталог.

### Числовая запись `chmod`

Каждое право имеет число: `r = 4`, `w = 2`, `x = 1`, отсутствие права — `0`. Числа складываются отдельно для владельца, группы и остальных:

- `7 = 4 + 2 + 1` → `rwx`;
- `6 = 4 + 2` → `rw-`;
- `5 = 4 + 1` → `r-x`;
- `4` → `r--`.

Например, `chmod 640 file` означает: владелец `6 = rw-`, группа `4 = r--`, остальные `0 = ---`. `chmod 755 script` означает `rwxr-xr-x`.

### Что означает `x` для директории

Директория **не является исполняемой программой**, но к ней применяются те же биты `r`, `w`, `x` с другим смыслом:

- `r` — прочитать список имён внутри директории;
- `w` — создавать, удалять и переименовывать записи внутри; обычно для этого также нужен `x`;
- `x` — пройти через директорию, выполнить `cd` и обратиться к известному объекту внутри по имени.

Каталог можно представить как таблицу `имя → inode`. Право `r` разрешает прочитать список имён из этой таблицы, а `x` — выполнить поиск имени и пройти к соответствующему объекту. Поэтому `r` без `x` почти бесполезен: простой `ls` иногда покажет имена, но `cd` не сработает, `ls -l` не сможет получить сведения, а файлы внутри нельзя будет открыть по пути.

Обратная комбинация `x` без `r` тоже имеет смысл: список скрыт, но к заранее известному имени можно обратиться, если права самого файла разрешают доступ. Для нормальной работы с каталогом обычно нужны вместе `r+x`; для создания и удаления записей — `w+x`. Право `x` требуется на каждой директории пути. Поэтому приватный каталог часто имеет `700 = rwx------`, а общий читаемый — `755 = rwxr-xr-x`.

`chmod` меняет права символически (`u=rw,go=r`) или числом (`640`). `stat -c '%a %n' file` показывает числовые права и имя. `chgrp` меняет группу файла, если пользователь имеет на это право. `chown` меняет владельца и обычно требует административных прав. Системные и чужие файлы изменять нельзя.

#### ❓ **Вопрос**: Какое числовое представление у прав `rwxr-x---`?

<details>

<summary><strong>Ответ</strong></summary>

`rwx = 7`, `r-x = 5`, `--- = 0`, значит `750`. Владелец: чтение, запись, выполнение; группа: чтение и выполнение; остальные — ничего.

</details>

In [ ]:
%%bash
whoami
id
groups
demo_file="${TMPDIR:-/tmp}/seminar1-files/result/report.txt"
mkdir -p "$(dirname "$demo_file")"
echo 'report' > "$demo_file"
ls -l "$demo_file"
chmod u=rw,go=r "$demo_file"
ls -l "$demo_file"
chmod 640 "$demo_file"
ls -l "$demo_file"
demo_directory="${TMPDIR:-/tmp}/seminar1-files/private-directory"
mkdir -p "$demo_directory"
chmod 700 "$demo_directory"
ls -ld "$demo_directory"
stat "$demo_file" | head -n 4

## 4. Процессы, jobs и сигналы

Процесс имеет PID и родительский PPID. `ps` показывает процессы системы, `jobs` — задачи текущей интерактивной оболочки. `&` запускает команду в фоне, `$!` содержит PID последнего фонового процесса, `wait` ожидает его завершения.

В интерактивном терминале `Ctrl+C` отправляет foreground-процессу SIGINT. `Ctrl+Z` приостанавливает его через SIGTSTP; `bg` продолжает задачу в фоне, `fg` возвращает на передний план. `kill PID` по умолчанию отправляет SIGTERM. SIGSTOP приостанавливает, SIGCONT продолжает. SIGKILL (`kill -9`) используют только в крайнем случае.

In [ ]:
%%bash
sleep 10 > /tmp/seminar1-sleep.out 2> /tmp/seminar1-sleep.err &
process_id=$!
echo "PID=$process_id"
ps -o pid,ppid,stat,cmd -p "$process_id"
kill -STOP "$process_id"
ps -o pid,ppid,stat,cmd -p "$process_id"
kill -CONT "$process_id"
kill -TERM "$process_id"
wait "$process_id" 2>/dev/null || true
ps -p "$process_id" || echo 'process finished'

## 5. Пайплайны

Оператор `|` направляет stdout команды слева в stdin команды справа. stderr в пайплайн автоматически не попадает. Пайплайн собирают по шагам: сначала проверяют каждую команду, затем соединяют.

По умолчанию код пайплайна равен коду последней команды. `set -o pipefail` делает пайплайн неуспешным, если завершилась с ошибкой любая его часть. `tee file` одновременно сохраняет вход в файл и передаёт его дальше.

In [ ]:
%%bash
set -o pipefail
cat /etc/os-release | head -n 5
ps -u "$USER" -o pid,stat,cmd | head -n 6
cat /etc/os-release | tee /tmp/os-release-copy.txt | wc -l
cat /missing-file | wc -l || echo 'pipeline failed'

## 6. Система и ресурсы

`uname` показывает kernel и архитектуру, `/etc/os-release` — дистрибутив. `uptime` показывает время работы и load average, `free` — память, `df` — место на файловых системах. `ps`, `top` и `htop` показывают процессы и нагрузку. `nvidia-smi` выводит состояние поддерживаемой NVIDIA GPU. Отсутствие `htop` или `nvidia-smi` нормально.

Фигурные скобки группируют несколько команд. Перенаправление после `}` применяется ко всей группе: обычный вывод всех команд попадёт в `system.txt`, ошибки — в `errors.txt`. После `{` нужен пробел или перенос строки, перед `}` — перевод строки или `;`. Команды выполняются последовательно в текущем Bash.

In [ ]:
%%bash
mkdir -p report
{
  uname -a
  cat /etc/os-release
  uptime
  free -h
  df -h "$HOME"
} > report/system.txt 2> report/errors.txt
cat report/system.txt
ps -u "$USER" -o pid,ppid,stat,%cpu,%mem,cmd --sort=-%mem 2>/dev/null | head -n 6 || ps -u "$USER" -o pid,ppid,stat,%cpu,%mem,cmd | head -n 6
command -v top >/dev/null && echo 'top: available' || echo 'top: not available'
command -v htop >/dev/null && echo 'htop: available' || echo 'htop: not available'
command -v nvidia-smi >/dev/null && echo 'nvidia-smi: available' || echo 'nvidia-smi: not available'

## Первый Bash-скрипт

Команды можно записать в обычный текстовый файл: Bash прочитает его сверху вниз и выполнит последовательно. Расширение `.sh` принято использовать для понятности, но Bash определяет запуск не по расширению.

Первая строка `#!/usr/bin/env bash` называется shebang. Она сообщает операционной системе, каким интерпретатором выполнять файл при прямом запуске. Строки после `#` являются комментариями. В этом скрипте нет условий, циклов и функций.

Скрипт создаётся одним heredoc. Маркер записан как `<<'EOF'`, поэтому `$USER` и `$HOME` не раскрываются при создании файла: они останутся внутри него и получат значения только при запуске.

In [ ]:
%%bash
script_file="${TMPDIR:-/tmp}/first_script.sh"
cat > "$script_file" <<'EOF'
#!/usr/bin/env bash
# Commands are executed from top to bottom
echo "Script started"
echo "USER=${USER}"
echo "HOME=${HOME}"
mkdir -p "$HOME/seminar1/script-result"
cd "$HOME/seminar1/script-result"
touch created-by-script.txt
echo "Created by Bash script" > created-by-script.txt
cp created-by-script.txt created-by-script-copy.txt
pwd
cat created-by-script-copy.txt
echo "Script finished"
EOF
cat "$script_file"

#### ❓ **Вопрос**: Файл `first_script.sh` содержит последовательность Bash-команд. Как его запустить?

<details>

<summary><strong>Ответ</strong></summary>

Двумя способами. Передать файл интерпретатору: `bash first_script.sh` — расширение и права при этом не важны. Либо сделать файл исполняемым и запустить напрямую: `chmod +x first_script.sh && ./first_script.sh`; для прямого запуска нужны право `x` и строка shebang `#!/usr/bin/env bash` в начале файла.

</details>